# 10 — Mind Map Generation

This notebook demonstrates the **MindMapWorkflow** — a graph-driven workflow that
extracts concept hierarchies from the knowledge graph and generates mind maps.

**Features:**
- Topic-level filtering (only nodes related to the topic)
- Depth control (limit tree depth)
- Node-edge JSON structure via Pydantic `MindMap` model
- Visualization with matplotlib

The mind map workflow is **graph-driven, not LLM-driven**. It extracts structure
from the existing knowledge graph and formats it as a `MindMap` model.

In [ ]:
import sys
sys.path.insert(0, '..')

from models import Concept, ConceptRelationship, Difficulty, MindMap, MindMapNode
from src.store.knowledge_graph import KnowledgeGraph
from src.workflows.mind_map import MindMapWorkflow

## 1. Build a Sample Knowledge Graph

We'll create a sample graph with concepts about Machine Learning to demonstrate
the mind map generation.

In [ ]:
# Create concepts
concepts = [
    Concept(id='ml-1', name='Machine Learning', definition='Field of AI that learns from data.',
            topics=['Machine Learning'], difficulty=Difficulty.MEDIUM, keywords=['ml', 'ai']),
    Concept(id='ml-2', name='Supervised Learning', definition='Learning with labeled data.',
            topics=['Machine Learning'], difficulty=Difficulty.MEDIUM, keywords=['supervised']),
    Concept(id='ml-3', name='Unsupervised Learning', definition='Learning without labels.',
            topics=['Machine Learning'], difficulty=Difficulty.MEDIUM, keywords=['unsupervised']),
    Concept(id='ml-4', name='Linear Regression', definition='Fitting a line to data.',
            topics=['Machine Learning'], difficulty=Difficulty.EASY, keywords=['regression']),
    Concept(id='ml-5', name='Decision Trees', definition='Tree-based classification.',
            topics=['Machine Learning'], difficulty=Difficulty.MEDIUM, keywords=['trees']),
    Concept(id='ml-6', name='K-Means Clustering', definition='Partition data into k clusters.',
            topics=['Machine Learning'], difficulty=Difficulty.MEDIUM, keywords=['clustering']),
    Concept(id='ml-7', name='Neural Networks', definition='Computing inspired by biology.',
            topics=['Machine Learning', 'Deep Learning'], difficulty=Difficulty.HARD, keywords=['nn', 'deep']),
    Concept(id='ml-8', name='Backpropagation', definition='Gradient computation for NNs.',
            topics=['Deep Learning'], difficulty=Difficulty.HARD, keywords=['backprop']),
    Concept(id='ml-9', name='Convolutional Neural Networks', definition='NNs for image data.',
            topics=['Deep Learning'], difficulty=Difficulty.HARD, keywords=['cnn', 'convolution']),
]

# Create relationships
relationships = [
    ConceptRelationship(source_concept='ml-1', target_concept='ml-2', relationship_type='prerequisite'),
    ConceptRelationship(source_concept='ml-1', target_concept='ml-3', relationship_type='prerequisite'),
    ConceptRelationship(source_concept='ml-2', target_concept='ml-4', relationship_type='prerequisite'),
    ConceptRelationship(source_concept='ml-2', target_concept='ml-5', relationship_type='prerequisite'),
    ConceptRelationship(source_concept='ml-3', target_concept='ml-6', relationship_type='prerequisite'),
    ConceptRelationship(source_concept='ml-1', target_concept='ml-7', relationship_type='related'),
    ConceptRelationship(source_concept='ml-7', target_concept='ml-8', relationship_type='prerequisite'),
    ConceptRelationship(source_concept='ml-7', target_concept='ml-9', relationship_type='prerequisite'),
]

# Build the knowledge graph
kg = KnowledgeGraph()
kg.add_concepts(concepts)
kg.add_relationships(relationships)

print(f'Knowledge graph: {len(kg)} nodes, {kg.graph.number_of_edges()} edges')

## 2. Generate a Mind Map

Initialize the workflow and generate a mind map for "Machine Learning" with default depth.

In [ ]:
workflow = MindMapWorkflow(knowledge_graph=kg)

# Generate mind map with default max_depth=3
mind_map = workflow.generate('Machine Learning', max_depth=3)

print(f'Mind map: {len(mind_map.nodes)} nodes, {len(mind_map.edges)} edges')
print()
print('Nodes:')
for node in mind_map.nodes:
    indent = '  ' if node.parent_id else ''
    print(f'{indent}[{node.type}] {node.label} (id={node.id})')

## 3. Inspect the JSON Structure

The MindMap model serializes to a clean JSON structure with nodes and edges.

In [ ]:
import json

# Pretty-print the JSON representation
print(json.dumps(mind_map.to_dict(), indent=2))

## 4. Depth Control

Demonstrate how `max_depth` controls the tree depth.

In [ ]:
for depth in [1, 2, 3]:
    result = workflow.generate('Machine Learning', max_depth=depth)
    print(f'max_depth={depth}: {len(result.nodes)} nodes, {len(result.edges)} edges')

## 5. Topic-Level Filtering

Generate a mind map for a sub-topic (Deep Learning) to see only relevant concepts.

In [ ]:
deep_learning_map = workflow.generate('Deep Learning', max_depth=3)

print(f'Deep Learning mind map: {len(deep_learning_map.nodes)} nodes')
for node in deep_learning_map.nodes:
    print(f'  [{node.type}] {node.label}')

## 6. Visualization

Visualize the mind map using matplotlib.

In [ ]:
%matplotlib inline

# Visualize the full Machine Learning mind map
workflow.visualize(mind_map)

In [ ]:
# Save visualization to file
workflow.visualize(mind_map, output_path='../outputs/mind_map_ml.png')
print('Saved to outputs/mind_map_ml.png')

## 7. Complex Topic Test

Test with a larger, more interconnected graph to verify handling of complex structures.

In [ ]:
# Add more concepts for a complex topic
complex_concepts = [
    Concept(id='dl-1', name='Activation Functions', definition='Non-linear transformations.',
            topics=['Deep Learning'], difficulty=Difficulty.MEDIUM, keywords=['relu', 'sigmoid']),
    Concept(id='dl-2', name='Loss Functions', definition='Measuring prediction error.',
            topics=['Deep Learning'], difficulty=Difficulty.MEDIUM, keywords=['loss', 'cost']),
    Concept(id='dl-3', name='Optimizers', definition='Algorithms to minimize loss.',
            topics=['Deep Learning'], difficulty=Difficulty.HARD, keywords=['sgd', 'adam']),
    Concept(id='dl-4', name='Regularization', definition='Preventing overfitting.',
            topics=['Deep Learning', 'Machine Learning'], difficulty=Difficulty.MEDIUM, keywords=['dropout', 'l2']),
    Concept(id='dl-5', name='Transfer Learning', definition='Reusing pre-trained models.',
            topics=['Deep Learning'], difficulty=Difficulty.HARD, keywords=['fine-tuning', 'pretrained']),
]

complex_rels = [
    ConceptRelationship(source_concept='ml-7', target_concept='dl-1', relationship_type='prerequisite'),
    ConceptRelationship(source_concept='ml-7', target_concept='dl-2', relationship_type='prerequisite'),
    ConceptRelationship(source_concept='dl-2', target_concept='dl-3', relationship_type='prerequisite'),
    ConceptRelationship(source_concept='ml-7', target_concept='dl-4', relationship_type='related'),
    ConceptRelationship(source_concept='ml-9', target_concept='dl-5', relationship_type='prerequisite'),
]

kg.add_concepts(complex_concepts)
kg.add_relationships(complex_rels)

# Generate expanded mind map
complex_map = workflow.generate('Deep Learning', max_depth=3)
print(f'Complex Deep Learning map: {len(complex_map.nodes)} nodes, {len(complex_map.edges)} edges')
print()
for node in complex_map.nodes:
    indent = '  ' * (0 if node.type == 'topic' else 1 if node.type == 'subtopic' else 2)
    print(f'{indent}[{node.type}] {node.label}')

In [ ]:
# Visualize the complex map
workflow.visualize(complex_map)

## Summary

The `MindMapWorkflow`:
1. **Filters** the knowledge graph to the specified topic
2. **Builds** a tree of `MindMapNode` objects via BFS with depth control
3. **Generates edges** between connected nodes (parent-child + graph relationships)
4. **Returns** a validated `MindMap` Pydantic model (serializable to JSON)
5. **Visualizes** the structure with matplotlib

This is entirely graph-driven — no LLM calls required.